In [1]:
import sys, os
sys.path.insert(0, r"PROJECT_ROOT")

from ts_init import pro
import pandas as pd
import numpy as np

# 加载带指标的股票数据
df_mt = pd.read_csv("../data/600519_with_indicators.csv")
df_mt["trade_date"] = pd.to_datetime(df_mt["trade_date"])
df_mt = df_mt.sort_values("trade_date").reset_index(drop=True)

print("✅ 茅台指标数据加载成功")
df_mt[["trade_date", "close", "ma5", "ma20", "rsi14"]].tail()

✅ Tushare 代理连接成功（30天试用账号）
✅ 茅台指标数据加载成功


,trade_date,close,ma5,ma20,rsi14
334,1970-01-01 00:00:00.020260526,1273.38,1295.092,1351.4770,2.997225
335,1970-01-01 00:00:00.020260527,1303.00,1292.692,1343.7025,24.564551
336,1970-01-01 00:00:00.020260528,1275.98,1285.688,1337.3415,19.464904
337,1970-01-01 00:00:00.020260529,1326.00,1292.848,1333.3915,41.042543
338,1970-01-01 00:00:00.020260601,1309.60,1297.592,1328.8130,39.133588


In [8]:
def check_portfolio_health(portfolio_dict):
    """
    portfolio_dict 示例：
    {
        "600519.SH": 0.6,   # 茅台占 60%
        "000300.SH": 0.4    # 沪深300占 40%
    }
    """
    report_lines = []
    total_weight = sum(portfolio_dict.values())
    
    for code, weight in portfolio_dict.items():
        # 这里简化：只用茅台的数据（你以后可以扩展成循环读多个 CSV）
        if code == "600519.SH":
            df = df_mt
        else:
            # 如果以后有别的股票，可以在这里加
            continue
            
        latest = df.iloc[-1]
        prev = df.iloc[-2]
        
        # 1. 检查 RSI 是否超买/超卖
        rsi = latest["rsi14"]
        if rsi > 70:
            rsi_status = "🔴 超买风险"
        elif rsi < 30:
            rsi_status = "🟢 超卖机会"
        else:
            rsi_status = "🟡 中性"
        
        # 2. 检查均线排列
        ma_status = "🟢 多头" if latest["ma5"] > latest["ma20"] else "🔴 空头"
        
        # 3. 检查仓位是否过重
        weight_warning = "⚠️ 仓位过重" if weight > 0.5 else "✅ 仓位合理"
        
        line = (
            f"{code} | 权重: {weight*100:.0f}% | "
            f"价格: {latest['close']:.2f} | "
            f"RSI: {rsi:.1f}({rsi_status}) | "
            f"趋势: {ma_status} | "
            f"{weight_warning}"
        )
        report_lines.append(line)
    
    # 汇总
    report_lines.append("\n📊 组合总权重: {:.0%}".format(total_weight))
    return "\n".join(report_lines)

In [3]:
# 假设你的持仓（你可以随时改这里的数字）
my_portfolio = {
    "600519.SH": 0.6,   # 60% 茅台
    "000300.SH": 0.4    # 40% 沪深300（虽然我们没算指标，但函数能跑）
}

report = check_portfolio_health(my_portfolio)
print(report)

600519.SH | 权重: 60% | 价格: 1309.60 | RSI: 39.1(🟡 中性) | 趋势: 🔴 空头 | ⚠️ 仓位过重

📊 组合总权重: 100%


In [7]:
import os
import joblib

# 确保 tools 文件夹存在
os.makedirs("../tools", exist_ok=True)

# 保存函数
joblib.dump(check_portfolio_health, "../tools/portfolio_checker.pkl")
print("✅ 持仓检查工具已保存，Agent 可以直接调用")

✅ 持仓检查工具已保存，Agent 可以直接调用
